## 3. Data Preparation — tratamento dos `-200` e imputação

Vamos comparar quatro alternativas:

1. mediana;
2. interpolação linear para lacunas de até 3 h + mediana;
3. KNN (`k=5`);
4. **interpolação curta (≤3 h) + KNN** para as lacunas restantes.

A avaliação mascara valores originalmente conhecidos em blocos de diferentes durações e calcula o **MAE normalizado pelo IQR** de cada variável. Menor é melhor.

In [ ]:
def median_impute(A):
    B = A.copy()
    medians = np.nanmedian(B, axis=0)
    ii = np.where(np.isnan(B))
    B[ii] = medians[ii[1]]
    return B


def interpolate_short_gaps(A, max_gap=3):
    B = A.copy()
    n, p = B.shape
    for j in range(p):
        col = B[:, j]
        i = 0
        while i < n:
            if not np.isnan(col[i]):
                i += 1
                continue
            start = i
            while i < n and np.isnan(col[i]):
                i += 1
            gap = i - start
            if (
                gap <= max_gap
                and start > 0
                and i < n
                and not np.isnan(col[start - 1])
                and not np.isnan(col[i])
            ):
                col[start:i] = np.linspace(col[start - 1], col[i], gap + 2)[1:-1]
        B[:, j] = col
    return B


def knn_impute(A):
    return KNNImputer(n_neighbors=5, weights='distance').fit_transform(A)


def short_linear_plus_median(A):
    return median_impute(interpolate_short_gaps(A, max_gap=3))


def short_linear_plus_knn(A):
    return knn_impute(interpolate_short_gaps(A, max_gap=3))


IMPUTERS = {
    'Mediana': median_impute,
    'Interpolação ≤3h + mediana': short_linear_plus_median,
    'KNN (k=5)': knn_impute,
    'Interpolação ≤3h + KNN': short_linear_plus_knn,
}

In [ ]:
def make_validation_mask(A, random_state=123):
    rng = np.random.default_rng(random_state)
    mask = np.zeros_like(A, dtype=bool)
    gap_plan = [(1, 12), (2, 10), (3, 8), (6, 6), (12, 4), (24, 2)]

    for j in range(A.shape[1]):
        used = np.zeros(A.shape[0], dtype=bool)
        for gap_len, target_blocks in gap_plan:
            candidates = [
                start for start in range(1, A.shape[0] - gap_len)
                if np.all(~np.isnan(A[start:start+gap_len, j]))
                and not np.isnan(A[start-1, j])
                and not np.isnan(A[start+gap_len, j])
            ]
            rng.shuffle(candidates)
            selected = 0
            for start in candidates:
                lo = max(0, start-1)
                hi = min(A.shape[0], start+gap_len+1)
                if used[lo:hi].any():
                    continue
                mask[start:start+gap_len, j] = True
                used[start:start+gap_len] = True
                selected += 1
                if selected >= target_blocks:
                    break
    return mask

validation_mask = make_validation_mask(X)
X_validation = X.copy()
X_validation[validation_mask] = np.nan

imputation_results = []
for name, function in IMPUTERS.items():
    imputed = function(X_validation)
    per_feature = []
    for j in range(X.shape[1]):
        m = validation_mask[:, j]
        q75, q25 = np.nanpercentile(X[:, j], [75, 25])
        scale = q75 - q25
        if not np.isfinite(scale) or scale == 0:
            scale = np.nanstd(X[:, j]) or 1.0
        nmae = np.mean(np.abs(imputed[m, j] - X[m, j])) / scale
        per_feature.append(nmae)
    imputation_results.append((name, float(np.mean(per_feature)), float(np.median(per_feature))))

imputation_results.sort(key=lambda x: x[1])

html = ['<table><tr><th>Método</th><th>NMAE médio</th><th>NMAE mediano</th></tr>']
for name, mean_err, median_err in imputation_results:
    html.append(f'<tr><td>{name}</td><td>{mean_err:.3f}</td><td>{median_err:.3f}</td></tr>')
html.append('</table>')
display(HTML(''.join(html)))

best_imputer_name = imputation_results[0][0]
best_imputer = IMPUTERS[best_imputer_name]
print('Método selecionado:', best_imputer_name) 

In [ ]:
X_imputed = best_imputer(X)
assert np.isfinite(X_imputed).all()

scaler = RobustScaler()
Z = scaler.fit_transform(X_imputed)

lo = np.percentile(Z, 0.5, axis=0)
hi = np.percentile(Z, 99.5, axis=0)
Z_cluster = np.clip(Z, lo, hi)

print('Matriz final:', Z.shape)